In [1]:
import numpy as np

# -----------------------------
# Observed "moments" from the letter (toy)
# -----------------------------
S = 109e9          # federal savings over 10 years ($)
U = 600_000        # uninsured increase (people)
# Note: These match the narrative 1.5m lose funding, 40% uninsured -> 0.6m uninsured.

# -----------------------------
# A simple welfare map
#   welfare = budget savings - (value loss per uninsured person)*uninsured
# interpret v as $ welfare loss per newly uninsured over 10y (can pick any)
# -----------------------------
def welfare(S, U, v):
    return S - v * U

# -----------------------------
# Compatibility mapping:
# given s (share kept covered), solve implied L and p from the two equations.
#   U = (1-s)L  =>  L = U/(1-s)
#   S = p L     =>  p = S/L
# This is "solving the fiber" after choosing s.
# -----------------------------
def implied_L_p(S, U, s):
    if s >= 1.0:
        return np.nan, np.nan
    L = U / (1.0 - s)
    p = S / L
    return L, p

# -----------------------------
# 1) Baseline: without pinning down s, the model is set-identified.
# We'll represent the identified set by scanning over plausible s values.
# -----------------------------
s_grid = np.linspace(0.05, 0.95, 2000)  # broad prior range just to illustrate
L_list, p_list = [], []
for s in s_grid:
    L, p = implied_L_p(S, U, s)
    if np.isfinite(L) and np.isfinite(p) and L > 0 and p > 0:
        L_list.append(L)
        p_list.append(p)

L_list = np.array(L_list)
p_list = np.array(p_list)

print("SET-ID baseline (vary s freely):")
print(f"  L range: {L_list.min()/1e6:.2f}m to {L_list.max()/1e6:.2f}m")
print(f"  p range: ${p_list.min():,.0f} to ${p_list.max():,.0f} per person (10y)")

# -----------------------------
# 2) Fiber thinning by a TEXT restriction:
#    (a) point restriction: "60% remain covered" => s=0.60
#    (b) bound restriction: "s in [0.50, 0.70]"
# -----------------------------
s_point = 0.60
L_point, p_point = implied_L_p(S, U, s_point)
print("\nTEXT: '60% remain covered' => s=0.60 (point thinning)")
print(f"  implied L = {L_point/1e6:.2f}m, implied p = ${p_point:,.0f} per person (10y)")

s_lo, s_hi = 0.50, 0.70
mask = (s_grid >= s_lo) & (s_grid <= s_hi)
L_band, p_band = [], []
for s in s_grid[mask]:
    L, p = implied_L_p(S, U, s)
    L_band.append(L); p_band.append(p)
L_band = np.array(L_band); p_band = np.array(p_band)
print("\nTEXT: 's in [0.50,0.70]' (bounded thinning)")
print(f"  L range: {L_band.min()/1e6:.2f}m to {L_band.max()/1e6:.2f}m")
print(f"  p range: ${p_band.min():,.0f} to ${p_band.max():,.0f} per person (10y)")

# -----------------------------
# 3) Welfare projection and "policy good" robustness
# Here welfare depends only on (S,U) given v. In this toy,
# S and U are taken as observed; to make welfare depend on assumptions,
# we let U be itself partly assumption-dependent, e.g. U = (1-s)L with L fixed by a separate source.
#
# So: suppose separate evidence pins down L ≈ 1.5m (as CBO states).
# Then varying s changes U and therefore welfare.
# -----------------------------
L_fixed = 1_500_000
# recompute U(s) and welfare(s) holding S fixed (savings) and allowing uninsured to vary with s
v = 50_000  # $ welfare loss per newly uninsured over 10y (illustrative)
s_grid2 = np.linspace(0.0, 0.95, 1000)
U_s = (1.0 - s_grid2) * L_fixed
w_s = welfare(S, U_s, v)

print("\nWelfare with L fixed at 1.5m and varying s (so U changes):")
print(f"  welfare range over s∈[0,0.95]: ${w_s.min()/1e9:.1f}B to ${w_s.max()/1e9:.1f}B")

# Robustness under a text-based bound on s
mask2 = (s_grid2 >= s_lo) & (s_grid2 <= s_hi)
w_band = w_s[mask2]
print(f"  welfare range over s∈[{s_lo},{s_hi}]: ${w_band.min()/1e9:.1f}B to ${w_band.max()/1e9:.1f}B")
print("  robustly 'policy good' (w>=0) over that s-band? ", w_band.min() >= 0)

SET-ID baseline (vary s freely):
  L range: 0.63m to 12.00m
  p range: $9,083 to $172,583 per person (10y)

TEXT: '60% remain covered' => s=0.60 (point thinning)
  implied L = 1.50m, implied p = $72,667 per person (10y)

TEXT: 's in [0.50,0.70]' (bounded thinning)
  L range: 1.20m to 2.00m
  p range: $54,559 to $90,792 per person (10y)

Welfare with L fixed at 1.5m and varying s (so U changes):
  welfare range over s∈[0,0.95]: $34.0B to $105.2B
  welfare range over s∈[0.5,0.7]: $71.5B to $86.5B
  robustly 'policy good' (w>=0) over that s-band?  True
